In [1]:
import os
import sys

# 1. Manually expose PyTorch's internal CUDA and CUDNN runtimes to the system linker
cuda_paths = [
    "/usr/local/cuda/lib64",
    "/usr/lib/x86_64-linux-gnu",
    # Maps internal python virtual env wheels if installed via pip
    os.path.join(sys.prefix, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages", "nvidia", "cublas", "lib"),
    os.path.join(sys.prefix, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages", "nvidia", "cudnn", "lib"),
]

# 2. Update the system LD_LIBRARY_PATH dynamically inside this runtime thread
existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(cuda_paths) + (f":{existing_ld}" if existing_ld else "")

print("Updated Dynamic Linker Paths Successfully.")

Updated Dynamic Linker Paths Successfully.


In [2]:
# !rm -rf /kaggle/working/*

In [3]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     # for filename in filenames:
#         print(os.path.join(dirname, ""))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
!mkdir -p /kaggle/working/nuscenes_test_root
!ln -s /kaggle/input/datasets/aadimator/nuscenes-test/Metadata/v1.0-test /kaggle/working/nuscenes_test_root/v1.0-test
!ln -s /kaggle/input/datasets/aadimator/nuscenes-test/Metadata/maps      /kaggle/working/nuscenes_test_root/maps
!ln -s "/kaggle/input/datasets/aadimator/nuscenes-test/Sensor file blobs/samples" /kaggle/working/nuscenes_test_root/samples
!ln -s "/kaggle/input/datasets/aadimator/nuscenes-test/Sensor file blobs/sweeps"  /kaggle/working/nuscenes_test_root/sweeps

In [5]:
!git clone -b kaggle-tiny-ft https://github.com/nitc-b23-pro/OpenEMMApro.git

Cloning into 'OpenEMMApro'...
remote: Enumerating objects: 675, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 675 (delta 119), reused 146 (delta 69), pack-reused 457 (from 1)
Receiving objects: 100% (675/675), 118.02 MiB | 50.35 MiB/s, done.
Resolving deltas: 100% (333/333), done.


In [6]:
%cd /kaggle/working/OpenEMMApro

/kaggle/working/OpenEMMApro


In [7]:
%ls

build_model.py       main.py              prompts.py                 utils.py
extract_nuscenes.py  openemma/            README.md
LICENSE              openemma_dataset.py  requirements.txt
llava-pythia/        policy_heads/        train_openemma_tinyvla.py


In [8]:
!cd /kaggle/working/OpenEMMApro && git pull

Already up to date.


In [9]:
# !git log --oneline --graph

In [10]:
pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 64.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      Successfully uninstalled opencv-python-4.13.0.92
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
   

In [11]:
pip install --no-deps nuscenes-devkit==1.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 6.7 MB/s eta 0:00:00ta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install cachetools descartes fire shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nuscenes-devkit 1.2.0 requires parameterized, which is not installed.
nuscenes-devkit 1.2.0 requires numpy<2.0.0,>=1.22.0, but you have numpy 2.0.2 which is incompatible.
nuscenes-devkit 1.2.0 requires Shapely~=2.0.3, but you have shapely 2.1.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [13]:
!python -c "import torch; print(torch.cuda.is_available())"

True


In [14]:
# import shutil

# # Replace 'your_folder_name' with the actual folder you want to delete
# folder_path = '/kaggle/working/OpenEMMApro'

# shutil.rmtree(folder_path)
# print("Folder deleted successfully!")

In [15]:
# import numpy
# import scipy
# import torch
# import torchvision
# import transformers
# print("numpy       :", numpy.__version__)
# print("scipy       :", scipy.__version__)
# print("torch       :", torch.__version__)
# print("torchvision :", torchvision.__version__)
# print("transformers:", transformers.__version__)

## Qwen

In [16]:
# import numpy, scipy, sklearn, transformers
# print(numpy.__version__, scipy.__version__, sklearn.__version__, transformers.__version__)
# from transformers import Qwen2_5_VLForConditionalGeneration
# print("import OK")

In [17]:
# import torch
# from transformers import Qwen2_5_VLForConditionalGeneration

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2/",
#     torch_dtype=torch.float16,
#     attn_implementation="sdpa",
#     device_map="auto"
# )

In [18]:
# import os, json

# src = "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2/"
# dst = "/kaggle/working/qwen2.5-vl-3b-fixed"
# os.makedirs(dst, exist_ok=True)

# for fname in os.listdir(src):
#     d = os.path.join(dst, fname)
#     if not os.path.exists(d):
#         os.symlink(os.path.join(src, fname), d)

# # First, inspect what's actually there
# cfg_path = os.path.join(dst, "preprocessor_config.json")
# with open(cfg_path) as f:
#     print(json.load(f))

In [19]:
# with open(cfg_path) as f:
#     cfg = json.load(f)

# os.remove(cfg_path)          # removes the symlink only, not your original input files
# cfg["image_processor_type"] = "Qwen2VLImageProcessor"
# cfg["processor_class"] = "Qwen2_5_VLProcessor"

# with open(cfg_path, "w") as f:
#     json.dump(cfg, f, indent=2)

In [20]:
# !python main.py --model-path lp --dataroot /kaggle/input/datasets/aadimator/nuscenes-mini/ --version v1.0-mini --method zero_shot

## LLava Pythia

In [21]:
!pip install peft==0.11.1 diffusers==0.27.2 einops einops-exts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.4 MB/s eta 0:00:0000:01
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1


In [22]:
!pip install -e ./llava-pythia --no-deps

Obtaining file:///kaggle/working/OpenEMMApro/llava-pythia
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava_pythia (pyproject.toml) ... done
  Created wheel for llava_pythia: filename=llava_pythia-1.0.0-0.editable-py3-none-any.whl size=3224 sha256=8982ac6cbac6f6031681db76d9a0609ed20b2007ea2d39c6fd639a7968812b37
  Stored in directory: /tmp/pip-ephem-wheel-cache-ha20nnwa/wheels/57/9f/98/bf8d4e816a205af27c8542ebc710ab3034c981319ab3cc5775
Successfully built llava_pythia


In [23]:
!pip install -e ./policy_heads

Obtaining file:///kaggle/working/OpenEMMApro/policy_heads
  Preparing metadata (setup.py) ... done
  Running setup.py develop for policy_heads


In [24]:
!ls

build_model.py	     main.py		  prompts.py		     utils.py
extract_nuscenes.py  openemma		  README.md
LICENSE		     openemma_dataset.py  requirements.txt
llava-pythia	     policy_heads	  train_openemma_tinyvla.py


In [25]:
# !python train_openemma_tinyvla.py

In [26]:
!python main.py --model-path lp --epoch /kaggle/input/models/latheeshpoondla/openemma-tinyvla-epoch11/other/default/1/ --dataroot /kaggle/input/datasets/aadimator/nuscenes-mini/ --version v1.0-mini --method openemma

/kaggle/working/OpenEMMApro/openemma/YOLO3D/utils/general.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are 

In [27]:
!zip -r /kaggle/working/OpenEMMApro/lp_results_ep19.zip /kaggle/working/OpenEMMApro/lp_results/openemma/20261002-100405

	zip warning: name not matched: /kaggle/working/OpenEMMApro/lp_results/openemma/20261002-100405

zip error: Nothing to do! (try: zip -r /kaggle/working/OpenEMMApro/lp_results_ep19.zip . -i /kaggle/working/OpenEMMApro/lp_results/openemma/20261002-100405)
